<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 3 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Model semantics, partitioning, and bucketing</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">Observe the results for identical keys under three models and understand how partitioning and bucketing narrow query ranges.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Target Doris 4.1.3 · Order data · Isolated lab database</span>
</div>

By the end, you will compare how three table models handle duplicate keys and observe date-partition and bucket pruning through query plans. Run the cells in order.

[Course notes](course3_models_partitioning_and_bucketing.md) · [Course home](../README.md)


## Lab scope

Only orders_duplicate, orders_unique, orders_aggregate, and orders_partitioned are rebuilt. Duplicate-key semantics and data distribution are two separate concerns.

The independent exercise rebuilds only orders_model_practice.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. Same keys, different models

Take order 1 (2300.00) and order 2 (405.00) from the WWI sample and simulate correcting order 1 to 2250.00 in the lab tables. After completing three submissions in order, observe:

- Duplicate Key retains all three input records.
- Unique Key retains two orders, with an amount of 2250.00 for order 1.
- Aggregate Key sums by key, producing 4550.00 for order 1.

The last result is suitable for additive metrics; to represent the current amount of an order after correction, use update-by-key semantics.


In [ ]:
definitions = {
    "orders_duplicate": 'CREATE TABLE orders_duplicate (id BIGINT, amount DECIMAL(12,2)) DUPLICATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
    "orders_unique": 'CREATE TABLE orders_unique (id BIGINT, amount DECIMAL(12,2)) UNIQUE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")',
    "orders_aggregate": 'CREATE TABLE orders_aggregate (id BIGINT, amount DECIMAL(12,2) SUM) AGGREGATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
}
for table, ddl in definitions.items():
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    show_sql("CREATE TABLE SQL", ddl)
    lab.execute(ddl)
    for row in [(1, "2300.00"), (1, "2250.00"), (2, "405.00")]:
        lab.insert(table, ["id", "amount"], [row])
expect(lab.query("SELECT id, amount FROM orders_duplicate ORDER BY id, amount"), [(1,"2250.00"),(1,"2300.00"),(2,"405.00")])
expect(lab.query("SELECT id, amount FROM orders_unique ORDER BY id"), [(1,"2250.00"),(2,"405.00")])
expect(lab.query("SELECT id, amount FROM orders_aggregate ORDER BY id"), [(1,"4550.00"),(2,"405.00")])
for table in definitions:
    lab.sql(f"SELECT * FROM {table} ORDER BY id, amount", title=table + ": Results for the same input")


## 2. What partitioning and bucketing each control

The partitioned table uses the original ten orders: one partition per day, with each partition Hash-bucketed by order ID into four buckets. Compare plans for a full-table query, a date-only filter, and a combined date and order ID filter in order, and observe how the selected partition and Tablet ranges narrow.

This table retains historical detail records, with date and order ID used for sorting; the business unique key for a current-order table must be designed separately.


In [ ]:
from dw_course.wwi import sample
lab.execute("DROP TABLE IF EXISTS orders_partitioned")
lab.execute("""
CREATE TABLE orders_partitioned (
    order_date DATE, order_id BIGINT, amount DECIMAL(18,2)
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p_day1 VALUES [('2013-01-01'), ('2013-01-02')),
    PARTITION p_day2 VALUES [('2013-01-02'), ('2013-01-03'))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 4
PROPERTIES("replication_num"="1")
""")
lab.insert("orders_partitioned", ["order_date", "order_id", "amount"],
           [(r["order_date"], r["order_id"], r["order_amount"]) for r in sample()["orders"]])
for query in [
    "SELECT * FROM orders_partitioned",
    "SELECT * FROM orders_partitioned WHERE order_date = '2013-01-01'",
    "SELECT * FROM orders_partitioned WHERE order_date = '2013-01-01' AND order_id = 1",
]:
    lab.sql("EXPLAIN " + query)
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM orders_partitioned WHERE order_date = '2013-01-01'"),
       [(5, "3944.20")])


## Completion criteria

Verify order counts and amounts for all three models, and find the selected partitions and Tablets in the EXPLAIN scan node. Focus on changes in scan range; field names and plan formatting may vary by version.


## Your turn: Which result is suitable for business use?

The Aggregate Key result of 4550.00 comes from 2300.00 + 2250.00. If a report needs the current amount of order 1, why is this result unsuitable? Then query the first day's amount in the partitioned table; it should be 3944.20 from the original sample.


## Independent exercise

The business needs to query the current amounts of orders. Receive (501,100), (501,80), and (502,40) in order, submitting each write only after the previous one completes. Choose a model for the separate table orders_model_practice, create it, and write the data; expect 80 for order 501 and 40 for order 502. Consider how the detail model and SUM aggregation would produce different results.

This exercise rebuilds only orders_model_practice.

Write and run your code in the next cell, then expand the reference solution after finishing. A blank exercise will not be automatically marked complete.


In [ ]:
# Write your SQL or load request here.


<details>
<summary>Reference solution (expand after completing the exercise)</summary>

```python
lab.execute("DROP TABLE IF EXISTS orders_model_practice")
lab.execute("""CREATE TABLE orders_model_practice (order_id BIGINT, amount DECIMAL(12,2))
UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")
lab.execute("INSERT INTO orders_model_practice VALUES (501, 100)")
lab.execute("INSERT INTO orders_model_practice VALUES (501, 80)")
lab.execute("INSERT INTO orders_model_practice VALUES (502, 40)")
lab.sql("SELECT * FROM orders_model_practice ORDER BY order_id", title="Current amounts of the two orders")
expect(lab.query("SELECT * FROM orders_model_practice ORDER BY order_id"), [(501,"80.00"),(502,"40.00")])
```

</details>
